In [ ]:
# !git clone https://github.com/william-darma-wijaya/thesis-2803.git
# %cd thesis-2803

In [ ]:
# !pip install -U bitsandbytes>=0.46.1
# !pip install sentence-transformers transformers networkx pandas torch tqdm nltk
#
# process_sql.py (dari external/spider_eval/, resmi SPIDER) butuh NLTK punkt tokenizer
# data untuk word_tokenize() -- ini TIDAK otomatis ter-download bareng pip install nltk.
# Coba download 'punkt' dan 'punkt_tab' (nama resource beda antar versi NLTK, coba dua-duanya).
# import nltk
# for _resource in ["punkt", "punkt_tab"]:
#     try:
#         nltk.download(_resource, quiet=True)
#     except Exception as _e:
#         print(f"[WARN] nltk.download({_resource!r}) gagal (mungkin tidak perlu di versi ini): {_e}")

# GraphRAG Text-to-SQL — Evaluation Pipeline

Notebook ini menjalankan dua pipeline Text-to-SQL secara berdampingan dan mengukur kualitas prediksi SQL menggunakan metrik resmi Spider benchmark.

⚠️ **Notebook ini adalah versi yang sudah disesuaikan ke struktur `src/` project (post-refactor foldering) dan ke implementasi evaluasi terbaru** (lihat `CLAUDE.md` dan `context/IMPLEMENTATION_DECISIONS.md` di root project untuk detail keputusan implementasi). Menjalankan notebook ini di Kaggle juga jadi **validasi end-to-end pertama** untuk kode evaluasi yang sebelumnya cuma diuji lewat sanity test di SQLite sintetis.

| Pipeline | Strategi Retrieval | Pruning |
|---|---|---|
| **GraphRAG** | Two-stage: table → column (n-gram + cosine) | Ya — hanya kolom relevan masuk prompt |
| **Baseline** | Single-stage: table-level (cosine) | Tidak — semua kolom tabel masuk prompt |

**Metrik yang diukur (per-query, in-process, bukan cuma agregat):**
- **ESM (Exact Set Match)** — apakah semua klausa SQL secara struktural identik dengan gold SQL
- **EX (Execution Accuracy)** — apakah hasil eksekusi SQL prediksi sama dengan gold SQL
- **CM (Component Match)** — skor ESM dipecah per klausa (SELECT/WHERE/GROUP BY/dst)
- **Difficulty** — easy/medium/hard/extra per query (konvensi resmi SPIDER, lihat `context/IMPLEMENTATION_DECISIONS.md` poin 6)
- **Gold Schema / Predicted Schema** — `table.column` yang dibutuhkan gold SQL vs yang di-retrieve pipeline
- **Token Input/Output** — untuk Token Consumption (T = T_in + α×T_out)
- **Schema Linking Recall/Precision** — seberapa baik retrieval memilih kolom yang relevan

Semua field di atas (kecuali recall/precision) disimpan sebagai `data/raw_logs/{graphrag,baseline}_log.json` sesuai skema `context/EVALUATION_ANALYSIS_GUIDE.md` Bagian 2 — jadi input siap pakai untuk `src/dimensions/` nanti, bukan cuma angka ringkasan di layar.

Evaluasi ESM/EX/CM dihitung **in-process** lewat `src/metrics/esm_ex_cm.py` (wrapper langsung ke `evaluation.py`/`process_sql.py` resmi SPIDER, BUKAN subprocess) supaya bisa dapat breakdown per-query per-klausa. Ada cross-check opsional pakai `evaluation.py` via subprocess di Bagian 10 untuk verifikasi kedua cara menghasilkan angka yang sama.

**Konfigurasi trial ini:** 50 sampel pertama dari Spider dev set.

## 1. Setup

Tambahkan root project ke Python path supaya modul `src.*` (`src.core`, `src.retrieval`, `src.generation`, `src.metrics`, `src.utils`, dll.) bisa diimport — cwd sudah jadi root project setelah `%cd thesis-2803` di cell pertama. Semua logika pipeline tetap berada di modul aslinya di `src/`; notebook ini hanya mengorkestrasikan.

In [ ]:
import sys
import os
import json
import re
import subprocess
from pathlib import Path

# Notebook ini ada di notebooks/eval_pipeline.ipynb, tapi setelah `%cd thesis-2803`
# di cell pertama, cwd adalah ROOT project -- semua modul dipanggil sebagai
# src.<paket>.<modul> (lihat CLAUDE.md), bukan flat import seperti versi lama
# notebook ini sebelum refactor foldering.
sys.path.insert(0, str(Path(".").resolve()))

from src.core.config import PipelineConfig
from src.core.schema import load_spider_schema, build_schema_graph
from src.retrieval.retrieval import build_schema_index
from src.retrieval.baseline import build_table_graph, build_table_index, run_single_baseline
from src.generation.few_shot import build_few_shot_index
from src.generation.generation import load_model_and_tokenizer
from src.experiments.pipeline import (
    run_single,
    _column_nodes_to_predicted_schema,
    _table_nodes_to_predicted_schema,
)

# Metrics yang diimplementasi di sesi evaluasi sebelumnya (lihat
# context/IMPLEMENTATION_DECISIONS.md) -- dipakai di sini supaya run Kaggle ini
# SEKALIGUS jadi validasi end-to-end pertama untuk kode itu terhadap Spider asli
# (sebelumnya cuma divalidasi lewat sanity test di SQLite sintetis).
from src.metrics import esm_ex_cm
from src.metrics.sla import extract_ground_truth_schema, compute_sla, aggregate_sla
from src.utils.schema_utils import load_db_schema
from src.utils.sql_normalize import normalize_sql_file_for_parsing

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

import pandas as pd

print("Imports OK")

## 2. Konfigurasi

`PipelineConfig` adalah sumber kebenaran tunggal untuk semua parameter — path data, model, hyperparameter retrieval, dan generasi. Ubah `SAMPLE_N` untuk mengontrol jumlah sampel yang dijalankan.

Parameter yang masih **TBD** (ditentukan setelah sweep/ablation):
- `top_k_tables`, `top_k_columns`, `semantic_similarity_threshold` → tunggu hasil `sweep.py`
- `few_shot_k` → tunggu hasil ablation study

In [ ]:
cfg = PipelineConfig()

SAMPLE_N = 50  # Jumlah sampel untuk trial. Ganti ke len(dev_data) untuk full dev set.

OUTPUT_DIR = Path("eval_output")
OUTPUT_DIR.mkdir(exist_ok=True)
RAW_LOGS_DIR = OUTPUT_DIR / "raw_logs"
RAW_LOGS_DIR.mkdir(exist_ok=True)

print(f"Data path         : {cfg.data_path}")
print(f"Embedding model   : {cfg.embedding_model}")
print(f"LLM model         : {cfg.llm_model}")
print(f"top_k_tables      : {cfg.top_k_tables}")
print(f"top_k_columns     : {cfg.top_k_columns}")
print(f"few_shot_k        : {cfg.few_shot_k}")
print(f"token_output_weight (α/μ in T = T_in + α×T_out): {cfg.token_output_weight}")
print(f"Sample N          : {SAMPLE_N}")
print(f"Output dir        : {OUTPUT_DIR.resolve()}")
print(f"Raw logs dir      : {RAW_LOGS_DIR.resolve()}")

## 3. Load Data (Spider Dev Set)

Spider dev set (`dev.json`) berisi 1034 pasang (question, gold SQL, db_id). Kita ambil `SAMPLE_N` sampel pertama.

Gold SQL juga ditulis ulang ke file `gold_subset.sql` dalam format yang diharapkan `evaluation.py`:
```
SELECT ...   <TAB>   db_id
```
File ini dipakai sebagai referensi saat menghitung ESM dan EX.

In [ ]:
with open(cfg.dev_json) as f:
    dev_data = json.load(f)

dev_subset = dev_data[:SAMPLE_N]

# Tulis gold SQL subset — format: {sql}\t{db_id} per baris
gold_path = OUTPUT_DIR / "gold_subset.sql"
with open(gold_path, "w", encoding="utf-8") as f:
    for item in dev_subset:
        f.write(item["query"] + "\t" + item["db_id"] + "\n")

print(f"Total dev set    : {len(dev_data)} samples")
print(f"Trial subset     : {len(dev_subset)} samples")
print(f"Gold SQL written : {gold_path}")
print()
print("Contoh sample pertama:")
ex = dev_subset[0]
print(f"  db_id    : {ex['db_id']}")
print(f"  question : {ex['question']}")
print(f"  gold SQL : {ex['query']}")

## 4. Bangun Schema Graph

Dua graph dibangun dari `tables.json` Spider dengan granularitas berbeda:

**GraphRAG graph** — node per kolom (`{db}.{table}.{col}`):
- Edge `belongs_to_pk`: kolom non-PK → PK dalam tabel yang sama
- Edge `foreign_key`: kolom FK → kolom PK di tabel lain
- Memungkinkan pruning di level kolom individu

**Baseline graph** — node per tabel (`{db}.{table}`):
- Edge `foreign_key`: tabel → tabel melalui relasi FK
- Retrieval hanya sampai level tabel, semua kolom tabel ikut masuk prompt

In [ ]:
print("Loading schema from tables.json...")
schema_df = load_spider_schema(cfg.tables_json)
print(f"Schema DataFrame: {len(schema_df)} rows")

print("\nBuilding GraphRAG column-level graph...")
graph = build_schema_graph(schema_df)
print(f"  Nodes: {graph.number_of_nodes():,}  |  Edges: {graph.number_of_edges():,}")

print("\nBuilding Baseline table-level graph...")
baseline_graph = build_table_graph(schema_df)
print(f"  Nodes: {baseline_graph.number_of_nodes():,}  |  Edges: {baseline_graph.number_of_edges():,}")

## 5. Load Models

Dua model diload sekali dan digunakan bersama oleh kedua pipeline:

| Model | Kegunaan | Catatan |
|---|---|---|
| `BAAI/bge-m3` | Schema linking + few-shot retrieval | Dipakai untuk encoding query dan schema |
| `Qwen/Qwen2.5-Coder-7B-Instruct` | SQL generation | 4-bit NF4 quantized, greedy decode |

Loading LLM membutuhkan ~5 menit dan ~6 GB VRAM. Pastikan GPU tersedia.

In [ ]:
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {cfg.embedding_model}")
embed_model = SentenceTransformer(cfg.embedding_model)
print("Embedding model loaded.")

print(f"\nLoading LLM: {cfg.llm_model} (4-bit quantized)...")
model, tokenizer = load_model_and_tokenizer(cfg)
print("LLM loaded.")

## 6. Bangun Index

**Few-shot index** — di-build sekali dari seluruh training set (~7000 pertanyaan). Untuk setiap dev question, retriever akan mencari pertanyaan training yang paling mirip secara semantik dan menyisipkannya ke prompt sebagai contoh.

**Schema index** — di-build lazy per database (hanya ketika pertama kali ditemukan dalam loop). Menyimpan embedding per kolom dan per tabel untuk keperluan schema linking.

**db_schema + kmaps (baru)** — dibutuhkan untuk evaluasi ESM/EX/CM/gold_schema per-query (`src/metrics/esm_ex_cm.py`, `src/metrics/sla.py`). `kmaps` (foreign-key map) di-build SEKALI untuk seluruh database di `tables.json` — jangan build ulang per query, lihat catatan performa di `esm_ex_cm.build_kmaps()`. `spider_schema_cache` (Schema object per db_id, untuk parsing SQL) di-cache lazy sama seperti schema/table index.

Build few-shot index membutuhkan beberapa menit karena mengkode seluruh training set.

In [ ]:
# Cache schema index per database — diisi lazy dalam loop
schema_cache = {}   # db_id -> SchemaIndex (untuk GraphRAG)
baseline_cache = {} # db_id -> TableSchemaIndex (untuk Baseline)

print("Building few-shot index from training set...")
few_shot_index = build_few_shot_index(cfg.train_json, embed_model)
print(f"Few-shot index built: {len(few_shot_index.examples)} training examples")

# --- Setup untuk evaluasi ESM/EX/CM/gold_schema per-query ---
print("\nLoading db_schema + building foreign-key maps for evaluation...")
db_schema = load_db_schema(str(cfg.tables_json))
kmaps = esm_ex_cm.build_kmaps(str(cfg.tables_json))
spider_schema_cache = {}  # db_id -> process_sql.Schema, di-cache lazy dalam loop

print(f"db_schema loaded for {len(db_schema)} databases")
print(f"kmaps built for {len(kmaps)} databases")

## 7. GraphRAG Inference Loop

Untuk setiap pertanyaan dalam dev subset, pipeline GraphRAG menjalankan:

1. **Stage 1** — pilih `top_k_tables` tabel kandidat via cosine similarity
2. **Stage 2** — dari kolom tabel kandidat, pilih `top_k_columns` kolom paling relevan per n-gram query
3. **Graph traversal** — cari shortest path antar kolom terpilih untuk menemukan jalur JOIN
4. **Path pruning** — buang intermediate node yang bukan PK/FK/direct hit
5. **Few-shot retrieval** — cari `few_shot_k` contoh training yang mirip
6. **Prompt + generate** — buat DDL prompt dan hasilkan SQL via Qwen2.5-Coder
7. **Evaluasi per-query (baru)** — ESM/EX/CM/difficulty (`esm_ex_cm.evaluate_single_query()`, in-process, bukan subprocess) + gold_schema (`extract_ground_truth_schema()`, SQL parser resmi SPIDER) + predicted_schema (dari column node yang di-retrieve) + token_input/token_output. Semua ini jadi satu entry `data/raw_logs/graphrag_log.json` sesuai skema di `context/EVALUATION_ANALYSIS_GUIDE.md` Bagian 2.

Schema index untuk setiap database di-cache agar tidak dibangun ulang jika ada beberapa pertanyaan dari database yang sama.

In [ ]:
graphrag_preds = []
graphrag_metrics = []
graphrag_raw_logs = []
graphrag_eval_results = []  # list of esm_ex_cm.QueryEvalResult, untuk aggregate_esm/ex/cm nanti

for i, item in enumerate(tqdm(dev_subset, desc="GraphRAG")):
    db_id = item["db_id"]
    gold_sql = item["query"]

    # Lazy-build schema index per database
    if db_id not in schema_cache:
        schema_cache[db_id] = build_schema_index(graph, db_id, embed_model)

    pred_sql, recall, precision, schema_ctx, col_nodes, n_in, n_out = run_single(
        question=item["question"],
        gold_sql=gold_sql,
        db_id=db_id,
        graph=graph,
        embed_model=embed_model,
        model=model,
        tokenizer=tokenizer,
        cfg=cfg,
        schema_index=schema_cache[db_id],
        few_shot_index=few_shot_index,
    )
    graphrag_preds.append(pred_sql)
    graphrag_metrics.append({"recall": recall, "precision": precision})

    # --- Evaluasi per-query (ESM/EX/CM/difficulty/gold_schema) untuk raw_logs ---
    # Try/except TERPISAH dari generation di atas -- kegagalan evaluasi (mis.
    # parser resmi SPIDER gagal, lihat context/IMPLEMENTATION_DECISIONS.md open
    # item soal LEFT JOIN) tidak boleh membuang prediksi yang sudah dihasilkan.
    try:
        db_path = str(cfg.db_dir / db_id / f"{db_id}.sqlite")
        if db_id not in spider_schema_cache:
            spider_schema_cache[db_id] = esm_ex_cm.build_schema_for_db(db_path)
        spider_schema = spider_schema_cache[db_id]
        kmap = kmaps[db_id]

        gold_schema = sorted(extract_ground_truth_schema(gold_sql, db_id, db_schema))
        eval_result = esm_ex_cm.evaluate_single_query(pred_sql, gold_sql, db_id, db_path, spider_schema, kmap)
        graphrag_eval_results.append(eval_result)

        graphrag_raw_logs.append({
            "query_id": f"dev_{i + 1:04d}", "db_id": db_id, "difficulty": eval_result.difficulty,
            "gold_sql": gold_sql, "predicted_sql": pred_sql,
            "gold_schema": gold_schema,
            "predicted_schema": _column_nodes_to_predicted_schema(graph, col_nodes),
            "token_input": n_in, "token_output": n_out,
            "esm_result": eval_result.esm, "ex_result": eval_result.ex,
            "cm_per_clause": eval_result.cm_per_clause,
        })
    except Exception as e:
        print(f"[WARN] Raw-log evaluation gagal untuk sample {i+1} (db={db_id}): {e!r}")

# Simpan prediksi ke file — satu SQL per baris
graphrag_pred_path = OUTPUT_DIR / "graphrag_predictions.txt"
graphrag_pred_path.write_text("\n".join(graphrag_preds), encoding="utf-8")

avg_recall = sum(m["recall"] for m in graphrag_metrics) / len(graphrag_metrics)
avg_prec   = sum(m["precision"] for m in graphrag_metrics) / len(graphrag_metrics)
print(f"\nGraphRAG selesai: {len(graphrag_preds)} sampel")
print(f"  Avg Schema Recall    : {avg_recall:.4f}")
print(f"  Avg Schema Precision : {avg_prec:.4f}")
print(f"  Predictions saved to : {graphrag_pred_path}")
print(f"  Raw-log entries      : {len(graphrag_raw_logs)} / {len(dev_subset)} (selisih = evaluasi gagal, lihat [WARN] di atas)")

## 8. Baseline Inference Loop

Pipeline Baseline menggunakan strategi yang lebih sederhana:

1. **Single-stage table linking** — embed query dan bandingkan langsung dengan embedding tabel (format `"table X columns Y Z..."`). Tidak ada two-stage atau n-gram matching.
2. **Table path tracing** — tambahkan tabel intermediate via FK jika ada gap konektivitas
3. **Schema context** — masukkan **semua kolom** dari setiap tabel terpilih ke prompt (tidak ada pruning)
4. **Zero-shot** — prompt tidak menyertakan few-shot examples (sesuai desain baseline)
5. **Evaluasi per-query (baru)** — sama seperti GraphRAG di atas, tapi predicted_schema-nya ALL kolom dari tabel yang terpilih (bukan cuma kolom yang di-retrieve), sesuai desain baseline "tidak ada pruning".

Perbedaan utama vs GraphRAG: token count prompt biasanya lebih tinggi karena tidak ada column-level pruning.

In [ ]:
baseline_preds = []
baseline_metrics = []
baseline_raw_logs = []
baseline_eval_results = []

for i, item in enumerate(tqdm(dev_subset, desc="Baseline")):
    db_id = item["db_id"]
    gold_sql = item["query"]

    # Lazy-build table index per database
    if db_id not in baseline_cache:
        baseline_cache[db_id] = build_table_index(baseline_graph, db_id, embed_model)

    pred_sql, recall, precision, table_nodes, n_in, n_out = run_single_baseline(
        question=item["question"],
        gold_sql=gold_sql,
        db_id=db_id,
        graph=baseline_graph,
        embed_model=embed_model,
        model=model,
        tokenizer=tokenizer,
        cfg=cfg,
        table_index=baseline_cache[db_id],
    )
    baseline_preds.append(pred_sql)
    baseline_metrics.append({"recall": recall, "precision": precision})

    # --- Evaluasi per-query (ESM/EX/CM/difficulty/gold_schema) untuk raw_logs ---
    try:
        db_path = str(cfg.db_dir / db_id / f"{db_id}.sqlite")
        if db_id not in spider_schema_cache:
            spider_schema_cache[db_id] = esm_ex_cm.build_schema_for_db(db_path)
        spider_schema = spider_schema_cache[db_id]
        kmap = kmaps[db_id]

        gold_schema = sorted(extract_ground_truth_schema(gold_sql, db_id, db_schema))
        eval_result = esm_ex_cm.evaluate_single_query(pred_sql, gold_sql, db_id, db_path, spider_schema, kmap)
        baseline_eval_results.append(eval_result)

        baseline_raw_logs.append({
            "query_id": f"dev_{i + 1:04d}", "db_id": db_id, "difficulty": eval_result.difficulty,
            "gold_sql": gold_sql, "predicted_sql": pred_sql,
            "gold_schema": gold_schema,
            "predicted_schema": _table_nodes_to_predicted_schema(baseline_graph, table_nodes),
            "token_input": n_in, "token_output": n_out,
            "esm_result": eval_result.esm, "ex_result": eval_result.ex,
            "cm_per_clause": eval_result.cm_per_clause,
        })
    except Exception as e:
        print(f"[WARN] Raw-log evaluation gagal untuk sample {i+1} (db={db_id}): {e!r}")

# Simpan prediksi ke file
baseline_pred_path = OUTPUT_DIR / "baseline_predictions.txt"
baseline_pred_path.write_text("\n".join(baseline_preds), encoding="utf-8")

avg_recall = sum(m["recall"] for m in baseline_metrics) / len(baseline_metrics)
avg_prec   = sum(m["precision"] for m in baseline_metrics) / len(baseline_metrics)
print(f"\nBaseline selesai: {len(baseline_preds)} sampel")
print(f"  Avg Schema Recall    : {avg_recall:.4f}")
print(f"  Avg Schema Precision : {avg_prec:.4f}")
print(f"  Predictions saved to : {baseline_pred_path}")
print(f"  Raw-log entries      : {len(baseline_raw_logs)} / {len(dev_subset)} (selisih = evaluasi gagal, lihat [WARN] di atas)")

## 9. Simpan Raw Logs + Ringkasan ESM/EX/CM/SLA (in-process)

Simpan `graphrag_raw_logs`/`baseline_raw_logs` ke JSON sesuai skema `context/EVALUATION_ANALYSIS_GUIDE.md` Bagian 2, lalu hitung ESM/EX/CM/SLA agregat langsung dari data yang sama — TANPA subprocess, angka ini datang dari evaluasi per-query yang sama yang mengisi raw_logs di atas.

⚠️ **SLA di sini beda dari "Schema Recall/Precision" yang sudah dicetak di Bagian 7/8.** Yang di Bagian 7/8 itu byproduct `evaluate_schema_linking()`/`evaluate_table_linking()` yang dipakai retrieval secara live cuma untuk progress print — token-matching, table+column dicampur jadi satu, TIDAK divalidasi terhadap `db_schema`. SLA di sini adalah metrik resmi guide 1.1: `gold_schema` diambil dari SQL parser resmi SPIDER (bukan token-matching), divalidasi, dan dihitung terpisah di level tabel & kolom. Dua angka ini TIDAK dijamin sama — kalau beda, itu SLA yang lebih dipercaya untuk pelaporan skripsi.

File disimpan ke `eval_output/raw_logs/` (bukan langsung ke `data/raw_logs/`) karena `SAMPLE_N` di notebook ini biasanya cuma sebagian dev set — biar tidak sengaja menimpa raw_logs "resmi" hasil full run. Kalau `SAMPLE_N` = `len(dev_data)` (full dev set) dan hasilnya sudah oke, salin manual ke `data/raw_logs/`.

In [ ]:
# --- Simpan raw_logs ---
graphrag_log_path = RAW_LOGS_DIR / "graphrag_log.json"
baseline_log_path = RAW_LOGS_DIR / "baseline_log.json"

with open(graphrag_log_path, "w", encoding="utf-8") as f:
    json.dump(graphrag_raw_logs, f, ensure_ascii=False, indent=2)
with open(baseline_log_path, "w", encoding="utf-8") as f:
    json.dump(baseline_raw_logs, f, ensure_ascii=False, indent=2)

print(f"Raw logs saved:")
print(f"  {graphrag_log_path}  ({len(graphrag_raw_logs)} entries)")
print(f"  {baseline_log_path}  ({len(baseline_raw_logs)} entries)")

# --- Ringkasan ESM/EX/CM in-process (dari graphrag_eval_results/baseline_eval_results) ---
print("\n" + "=" * 60)
print("ESM / EX (in-process, dari esm_ex_cm.evaluate_single_query)")
print("=" * 60)
print(f"{'':20} {'GraphRAG':>12} {'Baseline':>12}")
print(f"{'ESM (%)':20} {esm_ex_cm.aggregate_esm(graphrag_eval_results):>12.2f} {esm_ex_cm.aggregate_esm(baseline_eval_results):>12.2f}")
print(f"{'EX (%)':20} {esm_ex_cm.aggregate_ex(graphrag_eval_results):>12.2f} {esm_ex_cm.aggregate_ex(baseline_eval_results):>12.2f}")

print("\n" + "=" * 60)
print("CM per klausa (%) -- None berarti klausa itu tidak muncul di sampel manapun")
print("=" * 60)
graphrag_cm = esm_ex_cm.aggregate_cm(graphrag_eval_results)
baseline_cm = esm_ex_cm.aggregate_cm(baseline_eval_results)
print(f"{'Clause':15} {'GraphRAG':>12} {'Baseline':>12}")
for clause in esm_ex_cm.CLAUSES:
    g_val = graphrag_cm[clause]
    b_val = baseline_cm[clause]
    g_str = f"{g_val:.2f}" if g_val is not None else "None"
    b_str = f"{b_val:.2f}" if b_val is not None else "None"
    print(f"{clause:15} {g_str:>12} {b_str:>12}")

# --- Ringkasan Token Consumption (T = T_in + α×T_out, α dari cfg.token_output_weight) ---
alpha = cfg.token_output_weight
g_tin  = [e["token_input"]  for e in graphrag_raw_logs]
g_tout = [e["token_output"] for e in graphrag_raw_logs]
b_tin  = [e["token_input"]  for e in baseline_raw_logs]
b_tout = [e["token_output"] for e in baseline_raw_logs]
g_t = [i + alpha * o for i, o in zip(g_tin, g_tout)]
b_t = [i + alpha * o for i, o in zip(b_tin, b_tout)]

import statistics
print("\n" + "=" * 60)
print(f"Token Consumption (T = T_in + {alpha}×T_out)")
print("=" * 60)
print(f"{'':20} {'GraphRAG':>12} {'Baseline':>12}")
print(f"{'mean T_in':20} {statistics.mean(g_tin):>12.1f} {statistics.mean(b_tin):>12.1f}")
print(f"{'mean T_out':20} {statistics.mean(g_tout):>12.1f} {statistics.mean(b_tout):>12.1f}")
print(f"{'mean T':20} {statistics.mean(g_t):>12.1f} {statistics.mean(b_t):>12.1f}")
print(f"{'median T':20} {statistics.median(g_t):>12.1f} {statistics.median(b_t):>12.1f}")

# --- SLA (Schema Linking Accuracy, guide 1.1) -- table-level & column-level ---
# gold_schema/predicted_schema sudah tersimpan per-query di raw_logs sebagai list
# "table.column" -- compute_sla() butuh set, dan HARUS dipanggil sekali PER QUERY
# lalu di-aggregate_sla() (macro-average), BUKAN pool semua schema jadi satu set
# besar dulu (itu jadi micro-average, salah -- lihat context/METRICS_EXPLAINED.md).
def _dataset_sla(raw_logs: list, level: str):
    per_query = [
        compute_sla(set(entry["gold_schema"]), set(entry["predicted_schema"]), level=level)
        for entry in raw_logs
    ]
    return aggregate_sla(per_query)

g_sla_table  = _dataset_sla(graphrag_raw_logs, "table")
g_sla_column = _dataset_sla(graphrag_raw_logs, "column")
b_sla_table  = _dataset_sla(baseline_raw_logs, "table")
b_sla_column = _dataset_sla(baseline_raw_logs, "column")

print("\n" + "=" * 60)
print("SLA (Schema Linking Accuracy) -- guide 1.1, parser-based + validated")
print("=" * 60)
print(f"{'':22} {'GraphRAG':>12} {'Baseline':>12}")
print("-- Table level --")
print(f"{'  Precision (%)':22} {g_sla_table.precision*100:>12.2f} {b_sla_table.precision*100:>12.2f}")
print(f"{'  Recall (%)':22} {g_sla_table.recall*100:>12.2f} {b_sla_table.recall*100:>12.2f}")
print(f"{'  F1 (%)':22} {g_sla_table.f1*100:>12.2f} {b_sla_table.f1*100:>12.2f}")
print("-- Column level --")
print(f"{'  Precision (%)':22} {g_sla_column.precision*100:>12.2f} {b_sla_column.precision*100:>12.2f}")
print(f"{'  Recall (%)':22} {g_sla_column.recall*100:>12.2f} {b_sla_column.recall*100:>12.2f}")
print(f"{'  F1 (%)':22} {g_sla_column.f1*100:>12.2f} {b_sla_column.f1*100:>12.2f}")
print("\n(Recall is the headline number here -- see context/METRICS_EXPLAINED.md)")

## 10. Cross-Check: Spider Official Eval via Subprocess (opsional)

ESM/EX "resmi" sudah dihitung in-process di Bagian 9 (`esm_ex_cm.evaluate_single_query()`, sama-sama pakai `evaluation.py`/`process_sql.py` resmi SPIDER, cuma dipanggil langsung sebagai fungsi Python, bukan lewat subprocess). Cell di bawah menjalankan `evaluation.py` SEKALI LAGI lewat subprocess CLI (mirip `run_official_evaluation()` di `pipeline.py`) — **murni sebagai cross-check**: kalau angka `all` dari subprocess ini beda jauh dari ringkasan in-process di Bagian 9, itu tanda ada bug di wiring `esm_ex_cm.py`, bukan sesuatu yang harus selalu dijalankan.

Bisa di-skip kalau sudah percaya dengan hasil Bagian 9. Output subprocess ini juga mencakup breakdown per difficulty level (easy/medium/hard/extra) yang tidak otomatis kelihatan dari ringkasan in-process.

⚠️ **Soal parser JOIN-keyword gap (`context/IMPLEMENTATION_DECISIONS.md` poin 11):** predictions/gold file di sini dinormalisasi dulu (`normalize_sql_file_for_parsing()`) sebelum dioper ke `evaluation.py` CLI, sama seperti `pipeline.py`/`ablation.py`. Untuk **ESM** (`--etype match`), normalisasi PENUH — angka in-process (Bagian 9) dan cross-check di sini seharusnya cocok persis untuk SEMUA join type. Untuk **EX** (`--etype exec`), CLI resmi tidak bisa memisahkan "teks untuk parsing" dari "teks untuk eksekusi" seperti jalur in-process bisa (satu file dipakai untuk keduanya) — jadi cuma `INNER`/`CROSS JOIN` yang dinormalisasi di sini (aman dieksekusi), sementara `LEFT`/`RIGHT`/`FULL JOIN` tetap gagal parse di jalur CLI ini. Kalau warning `[WARN] EX ... beda > 1pp` muncul DAN prediksi mengandung `LEFT`/`RIGHT`/`FULL JOIN`, itu bukan bug wiring baru — itu residual limitation yang sudah didokumentasikan, bukan sesuatu yang perlu "diperbaiki" di notebook ini.

In [ ]:
def run_spider_eval(pred_file: Path, gold_file: Path, db_dir: Path,
                    tables_json: Path, etype: str) -> str:
    """Panggil evaluation.py Spider dan kembalikan stdout-nya.

    Menormalisasi predictions/gold file dulu lewat normalize_sql_file_for_parsing()
    sebelum dioper ke CLI -- process_sql.py resmi SPIDER cuma mengenali join-type
    keyword 'join' polos, predicted/gold SQL yang pakai INNER/CROSS/LEFT/RIGHT/
    FULL JOIN bikin get_sql() crash di dalam evaluation.py CLI (lihat
    context/IMPLEMENTATION_DECISIONS.md poin 11).

    etype='match' TIDAK PERNAH mengeksekusi SQL -- aman dinormalisasi PENUH
    (execution_safe_only=False). etype='exec' BENAR-BENAR mengeksekusi string
    dari file ini (CLI resmi tidak punya pemisahan raw-vs-normalized seperti
    jalur in-process esm_ex_cm.py) -- WAJIB execution_safe_only=True, cuma
    INNER/CROSS JOIN yang dinormalisasi (100% setara hasil eksekusi).
    LEFT/RIGHT/FULL JOIN karena itu tetap gagal parse di jalur exec ini --
    known residual limitation, lihat markdown Bagian 10 di atas.
    """
    execution_safe_only = (etype == "exec")
    norm_dir = pred_file.parent
    norm_gold = normalize_sql_file_for_parsing(
        gold_file, norm_dir / f"_normalized_{etype}_{gold_file.stem}.sql",
        has_db_id_suffix=True, execution_safe_only=execution_safe_only,
    )
    norm_pred = normalize_sql_file_for_parsing(
        pred_file, norm_dir / f"_normalized_{etype}_{pred_file.stem}.txt",
        has_db_id_suffix=False, execution_safe_only=execution_safe_only,
    )

    result = subprocess.run(
        [
            sys.executable, "external/spider_eval/evaluation.py",
            "--gold", str(norm_gold),
            "--pred", str(norm_pred),
            "--db",   str(db_dir),
            "--table", str(tables_json),
            "--etype", etype,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"[WARN] evaluation.py returned non-zero: {result.returncode}")
        print(result.stderr[-500:])
    return result.stdout


print("=" * 60)
print("GraphRAG — Exact Set Match")
print("=" * 60)
graphrag_esm_out = run_spider_eval(
    graphrag_pred_path, gold_path, cfg.db_dir, cfg.tables_json, "match")
print(graphrag_esm_out)

print("=" * 60)
print("GraphRAG — Execution Accuracy")
print("=" * 60)
graphrag_ex_out = run_spider_eval(
    graphrag_pred_path, gold_path, cfg.db_dir, cfg.tables_json, "exec")
print(graphrag_ex_out)

print("=" * 60)
print("Baseline — Exact Set Match")
print("=" * 60)
baseline_esm_out = run_spider_eval(
    baseline_pred_path, gold_path, cfg.db_dir, cfg.tables_json, "match")
print(baseline_esm_out)

print("=" * 60)
print("Baseline — Execution Accuracy")
print("=" * 60)
baseline_ex_out = run_spider_eval(
    baseline_pred_path, gold_path, cfg.db_dir, cfg.tables_json, "exec")
print(baseline_ex_out)

## 11. Ringkasan Hasil

Tabel berikut merangkum semua metrik untuk kedua pipeline. Kolom **ESM/EX** memakai angka in-process dari Bagian 9 (sumber utama, dan yang sama persis dipakai untuk mengisi `raw_logs`); kolom `_cross-check` di sampingnya adalah angka dari subprocess `evaluation.py` resmi di Bagian 10, buat verifikasi keduanya cocok.

- **Schema Recall/Precision** — angka LEGACY dari retrieval-time (`evaluate_schema_linking()`/`evaluate_table_linking()`), token-matching, table+column dicampur, TIDAK divalidasi. Bukan metrik resmi guide.
- **SLA** (tabel terpisah di bawah) — metrik resmi guide 1.1, parser-based, divalidasi terhadap `db_schema`, dipecah table-level vs column-level. Pakai ini untuk pelaporan skripsi, bukan Schema Recall/Precision di atas.
- **Mean T** — token consumption (T = T_in + α×T_out, lihat Bagian 9)

GraphRAG diharapkan memiliki **ESM/EX yang lebih tinggi** (retrieval lebih presisi), **SLA Recall yang tinggi di dua kondisi** dengan **SLA Precision lebih tinggi di GraphRAG** (sedikit kolom yang masuk = lebih tepat sasaran), dan **Mean T lebih rendah** (context lebih ringkas) dibanding Baseline.

In [ ]:
def parse_score(output: str, etype: str) -> float:
    """Ekstrak skor 'all' dari stdout evaluation.py."""
    # Cari baris yang diawali 'execution' (untuk exec) atau 'exact match' (untuk match)
    key = "execution" if etype == "exec" else "exact match"
    for line in output.split("\n"):
        if line.strip().lower().startswith(key):
            nums = re.findall(r"\d+\.\d+", line)
            if nums:
                return float(nums[-1])  # kolom terakhir = 'all'
    return float("nan")


results = {
    "Pipeline": ["GraphRAG", "Baseline"],
    "ESM (in-process)": [
        esm_ex_cm.aggregate_esm(graphrag_eval_results),
        esm_ex_cm.aggregate_esm(baseline_eval_results),
    ],
    "ESM (cross-check)": [
        parse_score(graphrag_esm_out, "match") * 100,
        parse_score(baseline_esm_out, "match") * 100,
    ],
    "EX (in-process)": [
        esm_ex_cm.aggregate_ex(graphrag_eval_results),
        esm_ex_cm.aggregate_ex(baseline_eval_results),
    ],
    "EX (cross-check)": [
        parse_score(graphrag_ex_out, "exec") * 100,
        parse_score(baseline_ex_out, "exec") * 100,
    ],
    "Schema Recall": [
        sum(m["recall"]    for m in graphrag_metrics) / len(graphrag_metrics),
        sum(m["recall"]    for m in baseline_metrics) / len(baseline_metrics),
    ],
    "Schema Precision": [
        sum(m["precision"] for m in graphrag_metrics) / len(graphrag_metrics),
        sum(m["precision"] for m in baseline_metrics) / len(baseline_metrics),
    ],
    "Mean T": [
        statistics.mean(g_t),
        statistics.mean(b_t),
    ],
}

df = pd.DataFrame(results).set_index("Pipeline")
display(df.round(4))

# Peringatan kalau in-process vs cross-check beda jauh -- indikasi bug wiring,
# bukan sekadar noise (dua-duanya pakai evaluator resmi SPIDER yang SAMA, jadi
# seharusnya identik atau sangat dekat, bukan cuma "mirip").
for label, in_proc, cross in [
    ("ESM GraphRAG", results["ESM (in-process)"][0], results["ESM (cross-check)"][0]),
    ("ESM Baseline", results["ESM (in-process)"][1], results["ESM (cross-check)"][1]),
    ("EX GraphRAG",  results["EX (in-process)"][0],  results["EX (cross-check)"][0]),
    ("EX Baseline",  results["EX (in-process)"][1],  results["EX (cross-check)"][1]),
]:
    if abs(in_proc - cross) > 1.0:  # toleransi 1 poin persen untuk floating point/rounding
        print(f"[WARN] {label}: in-process ({in_proc:.2f}) vs cross-check ({cross:.2f}) beda > 1pp -- cek wiring esm_ex_cm.py")

print()
print(f"CM per klausa (%) -- 'None' = klausa tidak muncul di sampel manapun:")
cm_df = pd.DataFrame({"GraphRAG": graphrag_cm, "Baseline": baseline_cm}).reindex(esm_ex_cm.CLAUSES)
display(cm_df)

print()
print("SLA (Schema Linking Accuracy) -- guide 1.1, per level (0-1 scale):")     
sla_df = pd.DataFrame({
    "GraphRAG (table)":   {"precision": g_sla_table.precision,  "recall": g_sla_table.recall,  "f1": g_sla_table.f1},
    "GraphRAG (column)":  {"precision": g_sla_column.precision, "recall": g_sla_column.recall, "f1": g_sla_column.f1},
    "Baseline (table)":   {"precision": b_sla_table.precision,  "recall": b_sla_table.recall,  "f1": b_sla_table.f1},
    "Baseline (column)":  {"precision": b_sla_column.precision, "recall": b_sla_column.recall, "f1": b_sla_column.f1},
})
display(sla_df.round(4))

print()
print(f"Total sampel dievaluasi: {SAMPLE_N}")
print(f"Output files:")
print(f"  {graphrag_pred_path}")
print(f"  {baseline_pred_path}")
print(f"  {gold_path}")
print(f"  {graphrag_log_path}")
print(f"  {baseline_log_path}")